**Eurovision Voting Blocs Across Eras (1975–2024)**

# LaTeX Tables for the Report

**Objective:** export the tables of the report as ready-to-use LaTeX files.

**Input:** the networks exported by notebook 02 (`data/02_network_construction/`) and the raw vote files (`data/raw/`)

**Output:** `data/02_network_construction/table_dataset_summary.tex`

> Run this notebook from the `src/` folder, after notebook 02.

In [ ]:
import os

import pandas as pd

from helper_functions import summary_statistics

## 1. Setup

In [ ]:
BASE_DIR = os.path.abspath("..")
DATA_RAW = os.path.join(BASE_DIR, "data", "raw")
DATA_PROC = os.path.join(BASE_DIR, "data", "02_network_construction")
DATA_ROBUST = os.path.join(DATA_PROC, "robustness")

In [ ]:
SUMMARY_ERAS = {
    "P1": {"span": "1975--1996", "desc": "Jury only", "y0": 1975, "y1": 1996},
    "P2": {"span": "1997--2008", "desc": "Televote rollout", "y0": 1997, "y1": 2008},
    "P3": {"span": "2009--2015", "desc": "50/50 Jury and Televote", "y0": 2009, "y1": 2015},
    "P4": {"span": "2016--2024", "desc": "Separate voting (combined)", "y0": 2016, "y1": 2024},
}

## 2. Dataset Summary Table

Descriptive statistics of the networks of each period, exported as `data/02_network_construction/table_dataset_summary.tex`.

- **Finals**: number of Grand Finals in the period;
- **N**: countries that voted or competed;
- **Weighted edges / mean W**: pairs with weight > 0 and their average weight;
- **Bin t ≥ 0.5 / t ≥ 0.33**: stable ties in the main and robustness binary networks;
- **Contig % / ComLang %**: share of country pairs that share a border / an official language.

### 2.1. Compute the Statistics

`summary_statistics` (in `helper_functions.py`) reads the networks exported by notebook 02 for each period and computes the columns described in the list above. The number of finals comes from `count_editions`, which counts the years with a Grand Final in the raw vote files.

In [ ]:
rows = summary_statistics(SUMMARY_ERAS, DATA_RAW, DATA_PROC, DATA_ROBUST)

In [ ]:
df_summary = pd.DataFrame(rows)
df_summary

### 2.2. Export the LaTeX Table

In [ ]:
# Generate LaTeX table
latex_lines = [
    r"\begin{table}[htbp]",
    r"\centering",
    r"\caption{Eurovision Voting Networks Dataset Summary across Eras (1975--2024)}",
    r"\label{tab:dataset_summary}",
    r"\small",
    r"\begin{tabular}{l l c c c c c c c c}",
    r"\hline\hline",
    r"Period & Voting System & Years & Finals & $N$ & $E_{\text{weighted}}$ & Mean $W$ & $E_{t \ge 0.5}$ & $E_{t \ge 0.33}$ & Contig. (\%) & ComLang (\%) \\",
    r"\hline",
]

for r in rows:
    line = (
        f"{r['Period']} & {r['Description']} & {r['Span']} & {r['Finals']} & {r['Nodes']} & "
        f"{r['WeightedEdges']} & {r['MeanWeight']:.3f} & {r['BinT05']} & {r['BinT033']} & "
        f"{r['ContigPct']:.1f}\\% & {r['ComLangPct']:.1f}\\% \\\\"
    )
    latex_lines.append(line)

latex_lines.extend([
    r"\hline\hline",
    r"\end{tabular}",
    r"\begin{tablenotes}",
    r"\footnotesize",
    r"\item \textit{Note:} $N$ represents active Grand Final participants/voters in the era; $E_{\text{weighted}}$ denotes directed ties where $W(A \to B) > 0$ for top-$k=3$; $E_{t \ge 0.5}$ and $E_{t \ge 0.33}$ are binarized edges at the main ($0.50$) and robustness ($0.33$) persistence thresholds requiring at least two top-$k$ voting occurrences ($\text{top-}k\text{ years} \ge 2$) to eliminate single-year artifact ties from late joiners; Contig. and ComLang show the percentage of directed dyads sharing a land border or official language from CEPII GeoDist.",
    r"\end{tablenotes}",
    r"\end{table}",
])

latex_content = "\n".join(latex_lines) + "\n"
latex_path = os.path.join(DATA_PROC, "table_dataset_summary.tex")
with open(latex_path, "w", encoding="utf-8") as f:
    f.write(latex_content)

print(f"\nLaTeX table successfully exported to: {latex_path}")